Insper

# Aula 09 - Spark ML - Machine Learning com Spark

Vamos fazer o setup de nosso ambiente Spark.

In [2]:
# Criar a sessao do Spark
from pyspark.sql import SparkSession
spark = SparkSession \
            .builder \
            .master("local[*]") \
            .appName("MichelSparkMLnov") \
            .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/17 23:03:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/11/17 23:03:06 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


## Modelo de regressão linear

Vamos usar o clássico conjunto de dados de Advertising para predizer o número de vendas de um produto, em função dos valores gastos em campanhas de TV, Radio e Jornal.

Iniciamos com a leitura do conjunto de dados.

In [3]:
data = spark.read.csv('../../dados/10_dados/ml_spark/Advertising.csv',
                      header=True,
                      inferSchema=True)

Exiba as primeiras cinco linhas do dataset.

In [4]:
data.show(5)

+---+-----+-----+---------+-----+
|_c0|   TV|Radio|Newspaper|Sales|
+---+-----+-----+---------+-----+
|  1|230.1| 37.8|     69.2| 22.1|
|  2| 44.5| 39.3|     45.1| 10.4|
|  3| 17.2| 45.9|     69.3|  9.3|
|  4|151.5| 41.3|     58.5| 18.5|
|  5|180.8| 10.8|     58.4| 12.9|
+---+-----+-----+---------+-----+
only showing top 5 rows



25/11/17 23:06:33 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , TV, Radio, Newspaper, Sales
 Schema: _c0, TV, Radio, Newspaper, Sales
Expected: _c0 but found: 
CSV file: file:///home/pads/notebooks/PADSONL07/dados/10_dados/ml_spark/Advertising.csv


Remova a coluna `_c0`.

In [5]:
data = data.drop("_c0")

Exiba novamente as primeiras cinco linhas.

In [6]:
data.show(5)

+-----+-----+---------+-----+
|   TV|Radio|Newspaper|Sales|
+-----+-----+---------+-----+
|230.1| 37.8|     69.2| 22.1|
| 44.5| 39.3|     45.1| 10.4|
| 17.2| 45.9|     69.3|  9.3|
|151.5| 41.3|     58.5| 18.5|
|180.8| 10.8|     58.4| 12.9|
+-----+-----+---------+-----+
only showing top 5 rows



Vamos agora exibir as principais estatísticas descritivas do conjunto de dados.

In [7]:
data.describe().show()

25/11/17 23:11:18 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+-----------------+------------------+------------------+------------------+
|summary|               TV|             Radio|         Newspaper|             Sales|
+-------+-----------------+------------------+------------------+------------------+
|  count|              200|               200|               200|               200|
|   mean|         147.0425|23.264000000000024|30.553999999999995|14.022500000000003|
| stddev|85.85423631490805|14.846809176168728| 21.77862083852283| 5.217456565710477|
|    min|              0.7|               0.0|               0.3|               1.6|
|    max|            296.4|              49.6|             114.0|              27.0|
+-------+-----------------+------------------+------------------+------------------+



In [8]:
data.dtypes

[('TV', 'double'),
 ('Radio', 'double'),
 ('Newspaper', 'double'),
 ('Sales', 'double')]

In [9]:
data.printSchema()

root
 |-- TV: double (nullable = true)
 |-- Radio: double (nullable = true)
 |-- Newspaper: double (nullable = true)
 |-- Sales: double (nullable = true)



O Spark faz uso de uma interface similar ao Scikit-Learn para desenvolver modelos preditivos. Baseado no conceito de `transformer`, nós vamos transformando o dataset em outro dataset com os ajustes necessários para o desenvolvimento de nosso modelo.

A estrutura de dados mais utilizada para o desenvolvimento de modelos é o `Vector` que possui, dentre outras funções, um bom suporte para dados esparsos.

Agora vamos importar o `VectorAssembler` que está em `pyspark.ml.feature` e também o modelo de regrssão linear (`LinearRegression`) que está em `pyspark.ml.regression`.

In [11]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression

Vamos dividir os dados no conjunto de treino (70%) e teste (30%), como segue:

In [12]:
train, test = data.randomSplit( [0.7, 0.3], seed=42 )

Para desenvolver seu modelo em Spark, você precisará ter uma coluna denominada `features`, que será resultado de todos os processos de transformação de dados necessários para o correto desenvolvimento dos modelos.

Vamos criar uma variável chamada `vec` que será o nosso `VectorAssembler`. Devemos informar quais colunas serão concatenadas em um `vector` e qual será o nome desse `vector`.

In [15]:
data.columns[:-1]

['TV', 'Radio', 'Newspaper']

Feito isso, podemos ver o resultado usando a função `transform()`.

In [18]:
vec = VectorAssembler(inputCols=data.columns[:-1],
                     outputCol="features")

In [19]:
train = vec.transform(train)
test = vec.transform(test)

Exiba as cinco primeiras linhas de train e, posteriormente, de teste.

In [20]:
train.show(5)

+---+-----+---------+-----+---------------+
| TV|Radio|Newspaper|Sales|       features|
+---+-----+---------+-----+---------------+
|0.7| 39.6|      8.7|  1.6| [0.7,39.6,8.7]|
|4.1| 11.6|      5.7|  3.2| [4.1,11.6,5.7]|
|7.3| 28.1|     41.4|  5.5|[7.3,28.1,41.4]|
|7.8| 38.9|     50.6|  6.6|[7.8,38.9,50.6]|
|8.4| 27.2|      2.1|  5.7| [8.4,27.2,2.1]|
+---+-----+---------+-----+---------------+
only showing top 5 rows



In [21]:
test.show(5)

+----+-----+---------+-----+----------------+
|  TV|Radio|Newspaper|Sales|        features|
+----+-----+---------+-----+----------------+
| 5.4| 29.9|      9.4|  5.3|  [5.4,29.9,9.4]|
| 8.6|  2.1|      1.0|  4.8|   [8.6,2.1,1.0]|
|11.7| 36.9|     45.2|  7.3|[11.7,36.9,45.2]|
|13.1|  0.4|     25.6|  5.3| [13.1,0.4,25.6]|
|17.2| 45.9|     69.3|  9.3|[17.2,45.9,69.3]|
+----+-----+---------+-----+----------------+
only showing top 5 rows



É possível observar a coluna `features`. Veja o schema do dataframe.

In [22]:
train.printSchema()

root
 |-- TV: double (nullable = true)
 |-- Radio: double (nullable = true)
 |-- Newspaper: double (nullable = true)
 |-- Sales: double (nullable = true)
 |-- features: vector (nullable = true)



Agora chegou a vez de criarmos o nosso modelo de regressão linear. Você precisa informar dois parâmetros: `featureCol` (qual é a coluna que possui um vector das features) e `labelCol` que representa a coluna que possui a variável dependente.

In [24]:
lr = LinearRegression(featuresCol="features",
                    labelCol="Sales")

Análogo ao `scikit-learn`, use o método `fit()`

In [25]:
lrModel = lr.fit(train)

25/11/17 23:25:41 WARN Instrumentation: [51463708] regParam is zero, which might cause numerical instability and overfitting.
25/11/17 23:25:41 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
25/11/17 23:25:42 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


O atributo `coefficients` e o atributo `intercept` apresentam, respectivamente, os coeficientes da regressão linear e o valor do intercepto.

In [26]:
type(lrModel)

pyspark.ml.regression.LinearRegressionModel

In [27]:
lrModel.coefficients

DenseVector([0.0459, 0.2009, -0.0035])

In [28]:
lrModel.intercept

2.735554921884285

E podemos usar a função `evaluate` para criar uma variável que nos permitirá obter as métricas comuns de desempenho do modelo.

In [29]:
evaluation_summary = lrModel.evaluate(test)

print("MAE: ", evaluation_summary.meanAbsoluteError)
print("RMSE: ", evaluation_summary.rootMeanSquaredError)
print("R2: ", evaluation_summary.r2)

MAE:  1.3365541938281158
RMSE:  1.6614200921535156
R2:  0.865104464823923


# Profissionalizando nossos modelos com Pipeline


De fato, há inúmeros procedimentos que fazemos com os dados antes de treinar um modelo. Há processos de limpeza, padronização, one hot encoding, etc.

Veja nesse link [https://spark.apache.org/docs/latest/ml-features.html](https://spark.apache.org/docs/latest/ml-features.html) as mais diversa funções que podemos fazer nos dados com o Spark. Há códigos de exemplo para lhe auxiliar no aprendizado.

Vamos comecar importando `Pipeline` que está em `pyspark.ml`. E com isso vamos definir um pipeline formado por 2 estágios: vec (transforma as features em vector) e lr (nosso modelo de regressão linear)

In [30]:
from pyspark.ml import Pipeline

E agora criamos nosso pipeline

In [31]:
pipeline = Pipeline( stages=[vec, lr] )

Vamos dividir novamente os dados em treino e teste

In [33]:
train, test = data.randomSplit( [0.7, 0.3], seed=42)

Agora podemos usar a função `fit()` do pipeline

In [34]:
pipelineModel = pipeline.fit(train)

25/11/17 23:54:38 WARN Instrumentation: [6bc98a64] regParam is zero, which might cause numerical instability and overfitting.


Execute a função `transform` sob o conjunto de teste (`test`) e salve em `pred`.

In [35]:
pred = pipelineModel.transform(test)

In [36]:
pred.show(5)

+----+-----+---------+-----+----------------+------------------+
|  TV|Radio|Newspaper|Sales|        features|        prediction|
+----+-----+---------+-----+----------------+------------------+
| 5.4| 29.9|      9.4|  5.3|  [5.4,29.9,9.4]| 8.958660646510957|
| 8.6|  2.1|      1.0|  4.8|   [8.6,2.1,1.0]| 3.548735670229504|
|11.7| 36.9|     45.2|  7.3|[11.7,36.9,45.2]| 10.52918016473767|
|13.1|  0.4|     25.6|  5.3| [13.1,0.4,25.6]|3.3276257934037177|
|17.2| 45.9|     69.3|  9.3|[17.2,45.9,69.3]|12.505787163829284|
+----+-----+---------+-----+----------------+------------------+
only showing top 5 rows



## Melhorando nosso pipeline com feature engineering

Lembre-se desse link para ver as mais usadas transformações nos dados: [https://spark.apache.org/docs/latest/ml-features.html](https://spark.apache.org/docs/latest/ml-features.html)


Vamos desenvolver outro modelo preditivo. Agora para predizer o gasto em planos de saúde. Teremos variáveis categóricas e contínuas.

Para as variáveis categóricas, vamos ver como aplicar `OneHotEncoding`.

In [37]:
gasto = spark.read.csv('../../dados/10_dados/ml_spark/gasto.csv', header=True, inferSchema=True)

Exiba as cinco primeiras linhas

In [38]:
gasto.show(5)

+---+------+------+--------+------+---------+-----------+
|age|   sex|   bmi|children|smoker|   region|    charges|
+---+------+------+--------+------+---------+-----------+
| 19|female|  27.9|       0|   yes|southwest|  16884.924|
| 18|  male| 33.77|       1|    no|southeast|  1725.5523|
| 28|  male|  33.0|       3|    no|southeast|   4449.462|
| 33|  male|22.705|       0|    no|northwest|21984.47061|
| 32|  male| 28.88|       0|    no|northwest|  3866.8552|
+---+------+------+--------+------+---------+-----------+
only showing top 5 rows



Exiba o schema.

In [39]:
gasto.printSchema()

root
 |-- age: integer (nullable = true)
 |-- sex: string (nullable = true)
 |-- bmi: double (nullable = true)
 |-- children: integer (nullable = true)
 |-- smoker: string (nullable = true)
 |-- region: string (nullable = true)
 |-- charges: double (nullable = true)



Separe agora em treino (70%) e teste (30%)

In [40]:
train, test = gasto.randomSplit( [0.7, 0.3], seed=42 )

Vamos agora definir as variáveis que são categóricas e quais são numéricas

In [43]:
categoricalCols = ["sex", "smoker", "region"]
numericalCols = ["age", "bmi", "children"]

Para fazer OneHotEncoder no Spark, primeiro precisamos transformar valores (`yes/no`) em números (`0/1`). Para isso temos que usar duas funções `StringIndexer` (que irá converter rótulos em valores) e posteriormente a `OneHotEncoder`. O resultado da `StringIndexer` é um vetor esparso. Você sabe dizer a utilidade disso?


Exemplo de um vetor esparso:

```
DenseVector(0, 0, 0, 7, 0, 2, 0, 0, 0, 0)
SparseVector(10, [3, 5], [7, 2])
```

In [44]:
from pyspark.ml.feature import OneHotEncoder, StringIndexer

Como sempre temos colunas como `input` e novas colunas como `output`, vamos definir o nome das colunas de output para cada uma das funções

In [45]:
indexOutputCols = [ x+"Index" for x in categoricalCols ]
indexOutputCols

['sexIndex', 'smokerIndex', 'regionIndex']

In [46]:
oheOutputCols = [ x+"OHE" for x in categoricalCols ]  
oheOutputCols

['sexOHE', 'smokerOHE', 'regionOHE']

Agora vamos criar nosso StringIndexer.

**Pergunta**: o que fazemos quando indexamos no treino e no teste há um valor desconhecido?

In [47]:
stringIndexer = StringIndexer(inputCols= categoricalCols,
                             outputCols= indexOutputCols,
                             handleInvalid= "skip")

E agora criamos nosso `OneHotEncoder`.

**Pergunta**: qual deve ser o input do OneHotEncoder?

In [65]:
oheEncoder = OneHotEncoder(inputCols= indexOutputCols, 
                          outputCols= oheOutputCols)

E também precisamos definir todas as colunas que serão usadas no vector que representará as features.

In [66]:
assemblerInputs = numericalCols + oheOutputCols
assemblerInputs

['age', 'bmi', 'children', 'sexOHE', 'smokerOHE', 'regionOHE']

Criamos o `VectorAssembler`

In [67]:
vecAssembler = VectorAssembler(inputCols= assemblerInputs,
                              outputCol= "variaveis_ml")

Vamos ver se você entendeu: Obtenha o resultado do oneEncoder para o conjunto de treino. Selecione apenas as 20 primeiras linhas, com as colunas `region`, `regionIndex` e `regionOHE` para verificar o resultado.

In [69]:
(oheEncoder.fit( stringIndexer.fit(train).transform(train) )
    .transform( stringIndexer.fit(train).transform(train) )
    .select("region", "regionIndex", "regionOHE")
    .show(20))

+---------+-----------+-------------+
|   region|regionIndex|    regionOHE|
+---------+-----------+-------------+
|southeast|        0.0|(3,[0],[1.0])|
|northeast|        1.0|(3,[1],[1.0])|
|northeast|        1.0|(3,[1],[1.0])|
|northeast|        1.0|(3,[1],[1.0])|
|southeast|        0.0|(3,[0],[1.0])|
|northeast|        1.0|(3,[1],[1.0])|
|northeast|        1.0|(3,[1],[1.0])|
|northeast|        1.0|(3,[1],[1.0])|
|southeast|        0.0|(3,[0],[1.0])|
|southeast|        0.0|(3,[0],[1.0])|
|northeast|        1.0|(3,[1],[1.0])|
|southeast|        0.0|(3,[0],[1.0])|
|southeast|        0.0|(3,[0],[1.0])|
|southeast|        0.0|(3,[0],[1.0])|
|southeast|        0.0|(3,[0],[1.0])|
|northeast|        1.0|(3,[1],[1.0])|
|southeast|        0.0|(3,[0],[1.0])|
|northeast|        1.0|(3,[1],[1.0])|
|northeast|        1.0|(3,[1],[1.0])|
|northeast|        1.0|(3,[1],[1.0])|
+---------+-----------+-------------+
only showing top 20 rows



Criamos agora nosso modelo de regressão linear

In [70]:
lr = LinearRegression(labelCol="charges",
                     featuresCol="variaveis_ml")

Criamos também nosso pipeline, composto pelos estágios:
- stringIndexer
- oheEncoder
- vecAssembler
- lr

In [71]:
pipeline = Pipeline( stages=[stringIndexer, oheEncoder, vecAssembler, lr] )

Aplicamos fit e, posteriormente, transform

In [73]:
pipelineModel = pipeline.fit(train)

25/11/18 00:47:19 WARN Instrumentation: [2e8352ff] regParam is zero, which might cause numerical instability and overfitting.


In [74]:
pred = pipelineModel.transform(test)

In [76]:
pred.select("age", "sex", "smoker", "variaveis_ml", "charges", "prediction" ).show(20)

+---+------+------+--------------------+-----------+------------------+
|age|   sex|smoker|        variaveis_ml|    charges|        prediction|
+---+------+------+--------------------+-----------+------------------+
| 18|female|    no|[18.0,24.09,1.0,0...|  2201.0971|   568.92382288308|
| 18|female|   yes|(8,[0,1,2,5],[18....| 18223.4512|25977.071474826116|
| 18|female|    no|(8,[0,1,4,6],[18....|7323.734819| 3049.623568745882|
| 18|female|    no|(8,[0,1,4,6],[18....| 2203.47185| 3383.684533991961|
| 18|female|    no|(8,[0,1,4,5],[18....|  1622.1885|2599.1639479576534|
| 18|female|    no|[18.0,31.35,4.0,0...|  4561.1885| 5908.719531404283|
| 18|female|    no|(8,[0,1,4,6],[18....|  2205.9808| 4018.400367959516|
| 18|female|    no|(8,[0,1,4,6],[18....| 2211.13075| 5321.238132419227|
| 18|female|   yes|(8,[0,1,5],[18.0,...| 36149.4835|27774.229338834422|
| 18|female|    no|(8,[0,1,4,5],[18....|  1631.6683|   4997.3700353032|
| 18|female|    no|(8,[0,1,4,5],[18....|  1631.8212| 5036.050778

Como avaliar agora um modelo que é um pipeline. para esse caso, temos que construir um objeto da classe `RegressionEvaluator`.

In [80]:
from pyspark.ml.evaluation import RegressionEvaluator

regEval = RegressionEvaluator(
    predictionCol= "prediction",
    labelCol= "charges",
    metricName= "rmse"
)

In [81]:
rmse = regEval.evaluate(pred)

print(f"RMSE é {rmse:.2f}")

RMSE é 5525.14


# Salvando o modelo

É uma boa prática salvar seu modelo para utilizar em outros momentos.

In [82]:
pipelineModel.write().overwrite().save("./lr-pipeline-model")

# Carregando o modelo

In [83]:
from pyspark.ml import PipelineModel

In [84]:
model = PipelineModel.load("./lr-pipeline-model")

In [85]:
model.stages

[StringIndexerModel: uid=StringIndexer_2029aba88ae8, handleInvalid=skip, numInputCols=3, numOutputCols=3,
 OneHotEncoderModel: uid=OneHotEncoder_96468e9fd5de, dropLast=true, handleInvalid=error, numInputCols=3, numOutputCols=3,
 VectorAssembler_886cb06ef2ba,
 LinearRegressionModel: uid=LinearRegression_bcc568762a9a, numFeatures=8]

# Otimização de hiperparâmetros e Validação Cruzada

Vamos construir um modelo para nosso problema em questão agora fazendo uso de um `RandomForestRegressor`. Vamos querer otimizar 2 hiperparâmetros (`maxDepth` e `numTrees`), e obter seus valores a partir de um processo de validação cruzada com 5-folds.

In [86]:
from pyspark.ml.regression import RandomForestRegressor

In [88]:
rf = RandomForestRegressor(labelCol="charges",
                         featuresCol="variaveis_ml",
                         seed=42)

In [89]:
pipeline_rf = Pipeline( stages=[stringIndexer, oheEncoder, vecAssembler, rf] )

Para executar o grid, precisamos criar um objeto da classe `ParamGridBuilder`

In [90]:
from pyspark.ml.tuning import ParamGridBuilder

In [91]:
paramGrid = (ParamGridBuilder()
            .addGrid(rf.maxDepth, [2, 4, 6])
            .addGrid(rf.numTrees, [10, 100])
            .build())

E criamos também um `RegressionEvaluator`

In [92]:
evaluator = RegressionEvaluator(
    labelCol="charges",
    predictionCol="prediction", 
    metricName="rmse"
)

Para executar a validação cruzada, nós precisamos importar a função `CrossValidator` de `pyspark.ml.tuning`.

In [93]:
from pyspark.ml.tuning import CrossValidator

In [94]:
cv = CrossValidator(
    estimator= pipeline_rf, 
    evaluator= evaluator, 
    estimatorParamMaps= paramGrid, 
    numFolds= 5,
    seed=42
)

E podemos obter nosso modelo a partir do `fit`.

In [95]:
cvModel = cv.fit(train)

25/11/18 01:16:54 WARN DAGScheduler: Broadcasting large task binary with size 1025.3 KiB
25/11/18 01:17:08 WARN DAGScheduler: Broadcasting large task binary with size 1030.3 KiB
25/11/18 01:17:20 WARN DAGScheduler: Broadcasting large task binary with size 1027.0 KiB
25/11/18 01:17:31 WARN DAGScheduler: Broadcasting large task binary with size 1036.6 KiB
25/11/18 01:17:43 WARN DAGScheduler: Broadcasting large task binary with size 1045.5 KiB
25/11/18 01:17:47 WARN DAGScheduler: Broadcasting large task binary with size 1022.7 KiB


E também podemos verificar todas as trials feitas no processo de otimização.

In [96]:
cvModel.avgMetrics

[np.float64(7260.499999810339),
 np.float64(7344.564015501814),
 np.float64(5790.160763194973),
 np.float64(5496.943055865067),
 np.float64(5227.464757906523),
 np.float64(4923.7763712791)]

In [97]:
cvModel.getEstimatorParamMaps()

[{Param(parent='RandomForestRegressor_208ea85211fb', name='maxDepth', doc='Maximum depth of the tree. (>= 0) E.g., depth 0 means 1 leaf node; depth 1 means 1 internal node + 2 leaf nodes. Must be in range [0, 30].'): 2,
  Param(parent='RandomForestRegressor_208ea85211fb', name='numTrees', doc='Number of trees to train (>= 1).'): 10},
 {Param(parent='RandomForestRegressor_208ea85211fb', name='maxDepth', doc='Maximum depth of the tree. (>= 0) E.g., depth 0 means 1 leaf node; depth 1 means 1 internal node + 2 leaf nodes. Must be in range [0, 30].'): 2,
  Param(parent='RandomForestRegressor_208ea85211fb', name='numTrees', doc='Number of trees to train (>= 1).'): 100},
 {Param(parent='RandomForestRegressor_208ea85211fb', name='maxDepth', doc='Maximum depth of the tree. (>= 0) E.g., depth 0 means 1 leaf node; depth 1 means 1 internal node + 2 leaf nodes. Must be in range [0, 30].'): 4,
  Param(parent='RandomForestRegressor_208ea85211fb', name='numTrees', doc='Number of trees to train (>= 1).

In [100]:
cvModel.bestModel.stages

[StringIndexerModel: uid=StringIndexer_2029aba88ae8, handleInvalid=skip, numInputCols=3, numOutputCols=3,
 OneHotEncoderModel: uid=OneHotEncoder_96468e9fd5de, dropLast=true, handleInvalid=error, numInputCols=3, numOutputCols=3,
 VectorAssembler_886cb06ef2ba,
 RandomForestRegressionModel: uid=RandomForestRegressor_208ea85211fb, numTrees=100, numFeatures=8]

In [105]:
cvModel.bestModel.stages[-1]._java_obj.getMaxDepth()

6

# Link para referência adicional

Vejam aqui um exemplo de um pipeline de **classificação**.

https://swan-gallery.web.cern.ch/notebooks/SparkTraining/notebooks/ML_Demo1_Classifier.html